In [ ]:
#1. Imports
# general imports
from pathlib import Path
import numpy as np
import pandas as pd
import re
from tqdm import tqdm

np.random.seed(42)

from nispace.datasets import fetch_reference
from nispace.plotting import view_surf
from nispace.workflows import group_comparison
import statsmodels.api as sm

In [ ]:
#2. Settings
DATA_PATH = Path("../../data/df1.csv")

DK_REGIONS = [
    "bankssts","caudalanteriorcingulate","caudalmiddlefrontal","cuneus","entorhinal",
    "fusiform","inferiorparietal","inferiortemporal","isthmuscingulate","lateraloccipital",
    "lateralorbitofrontal","lingual","medialorbitofrontal","middletemporal","parahippocampal",
    "paracentral","parsopercularis","parsorbitalis","parstriangularis","pericalcarine",
    "postcentral","posteriorcingulate","precentral","precuneus","rostralanteriorcingulate",
    "rostralmiddlefrontal","superiorfrontal","superiorparietal","superiortemporal",
    "supramarginal","frontalpole","temporalpole","transversetemporal","insula",
]

DEMO_COLS =["PATNO", "CONCOHORT", "age", "SEX", "agediag", "subgroup", "PRIMDIAG"]

CONTRASTS = {
    "De Novo PD vs HC": (1.0, 2.0),
    "Prodromal PD vs HC": (4.0, 2.0),
    "De Novo PD vs Prodromal PD": (1.0, 4.0)
}

GROUP_LABELS = {
    1.0: "De Novo PD ",
    2.0: "HC",
    4.0: "Prodromal PD"
}


In [ ]:
#Choosing the reference maps
#Going by partial matching - MIGHT CHANGE
SELECTED_REFERENCE_MAPS = [
"target-mGluR5_tracer-abp688_n-73_dx-hc_pub-smart2019",
"target-NMDA_tracer-ge179_n-29_dx-hc_pub-galovic2021",
"target-GABAa_tracer-flumazenil_n-6_dx-hc_pub-dukart2018",
"target-FDOPA_tracer-fluorodopa_n-12_dx-hc_pub-garciagomez2018",
"target-D1_tracer-sch23390_n-13_dx-hc_pub-kaller2017",
"target-D23_tracer-flb457_n-55_dx-hc_pub-sandiego2015",
"target-DAT_tracer-fpcit_n-174_dx-hc_pub-dukart2018",
"target-5HT1a_tracer-way100635_n-35_dx-hc_pub-savli2012",
"target-5HT1b_tracer-p943_n-23_dx-hc_pub-savli2012",
"target-5HT2a_tracer-altanserin_n-19_dx-hc_pub-savli2012",
"target-5HT4_tracer-sb207145_n-59_dx-hc_pub-beliveau2017",
"target-5HTT_tracer-dasb_n-18_dx-hc_pub-savli2012",
"target-NET_tracer-mrb_n-10_dx-hc_pub-hesse2017",
"target-VAChT_tracer-feobv_n-18_dx-hc_pub-aghourian2017",
]

SELECTED_REFERENCE_CONTAINS = []


N_PERM = 10000

In [ ]:
#3. Helper functions
def get_dk_thickness_columns(df: pd.DataFrame) -> list[str]:
    """Return DK cortical thickness columns only"""
    dk_set = {r.lower() for r in DK_REGIONS}
    result = []
    for col in df.columns:
        col_lower = col.lower()
        if not col_lower.endswith("_thickness"):
            continue
        for hemi in ("lh_", "rh_"):
            if col_lower.startswith(hemi):
                region = col_lower[len(hemi):-len("_thickness")]
                if region in dk_set:
                    result.append(col)
                break
    return result

def rename_to_nispace(col: str) -> str:
    """Convert lh/rh thickness column names to NiSpace DK parcel names"""
    match = re.match(r"(lh|rh)_(.+)_thickness", col)
    if not match:
        return col

    hemi, region = match.groups()
    hemi_letter = "L" if hemi == "lh" else "R"
    return f"hemi-{hemi_letter}_lab-{region}"

def prepare_brain_and_design(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Prepare Y matrix and design matrix"""
    demo_cols = [c for c in DEMO_COLS if c in df.columns]
    dk_cols = get_dk_thickness_columns(df)

    print("Number of DK thickness columns:", len(dk_cols))
    print("Has MeanThickness columns?", any ("MeanThickness" in c for c in dk_cols))

    extra_cols = [c for c in ["Field Strength"] if c in df.columns]
    df_dk = df[demo_cols + extra_cols + dk_cols].copy()

    #brain matrix
    Y = df_dk[dk_cols].copy()
    Y.index = df_dk["PATNO"]

    #design matrix
    design = df_dk[["CONCOHORT", "age", "SEX", "Field Strength"]].copy()
    design.index = df_dk["PATNO"]
    design["CONCOHORT"] = pd.to_numeric(design["CONCOHORT"], errors="coerce")
    design["Field Strength"] = pd.to_numeric(design["Field Strength"], errors="coerce")

    return Y, design

def align_y_to_reference(Y: pd.DataFrame, ref_df: pd.DataFrame) -> pd.DataFrame:
    """Rename and reorder Y columns to match reference parcel columns"""
    Y_renamed = Y.copy()
    Y_renamed.columns = [rename_to_nispace(c) for c in Y.columns]

    Y_aligned = Y_renamed.reindex(columns=ref_df.columns).copy()

    print("Y aligned shape:", Y_aligned.shape)
    print("Any missing values after parcel reindex?", Y_aligned.isna().any().any())

    missing_cols = Y_aligned.columns[Y_aligned.isna().all(axis=0)].tolist()
    if missing_cols:
        print("Warning: these parcels are entirely missing in Y:")
        for col in missing_cols:
            print(" -", col)

    return Y_aligned

def run_parcelwise_ttest(
        Y_aligned: pd.DataFrame,
        design: pd.DataFrame,
        g1: float,
        g2: float,
) -> pd.DataFrame:
    """
    Parcelwise group comparison adjusted for age, SEX, and Field Strength using OLS:
        thickness ~ group + age + SEX + Field_Strength

    With coding {g1: 0, g2: 1}, a positive t-value means higher adjusted thickness in g2.
    """
    mask = design["CONCOHORT"].astype(float).isin([g1, g2])
    y_sub = Y_aligned.loc[mask].copy()
    d_sub = design.loc[mask].copy()

    d_sub = d_sub.copy()
    d_sub["group01"] = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1})

    if d_sub["SEX"].dtype == object:
        d_sub["SEX"] = pd.Categorical(d_sub["SEX"]).codes
    d_sub["SEX"] = pd.to_numeric(d_sub["SEX"], errors="coerce")
    d_sub["age"] = pd.to_numeric(d_sub["age"], errors="coerce")
    d_sub["Field Strength"] = pd.to_numeric(d_sub["Field Strength"], errors="coerce")
    d_sub["group01"] = pd.to_numeric(d_sub["group01"], errors="coerce")

    t_vals, p_vals, dfs = [], [], []

    for parcel in y_sub.columns:
        tmp = pd.DataFrame(
            {
                "y": pd.to_numeric(y_sub[parcel], errors="coerce"),
                "group01": d_sub["group01"],
                "age": d_sub["age"],
                "SEX": d_sub["SEX"],
                "Field Strength": d_sub["Field Strength"],
            },
            index=y_sub.index,
        ).dropna()

        if tmp.shape[0] < 5 or tmp["group01"].nunique() < 2:
            t_vals.append(np.nan)
            p_vals.append(np.nan)
            dfs.append(np.nan)
            continue

        X = sm.add_constant(tmp[["group01", "age", "SEX", "Field Strength"]])
        model = sm.OLS(tmp["y"], X).fit()

        t_vals.append(model.tvalues.get("group01", np.nan))
        p_vals.append(model.pvalues.get("group01", np.nan))
        dfs.append(model.df_resid)

    return pd.DataFrame(
        {
            "Tvalue": t_vals,
            "pvalue": p_vals,
            "df": dfs,
            "hemi": ["L" if c.startswith("hemi-L") else "R" for c in Y_aligned.columns],
        },
        index=Y_aligned.columns,
    )

def build_nispace_design(d_sub: pd.DataFrame, g1: float, g2: float) -> pd.DataFrame:
    """
    Build design DataFrame for NiSpace:
    groups: 0 = g1, 1 = g2
    covariates: age, SEX, Field Strength
    """
    groups01 = d_sub["CONCOHORT"].astype(float).map({g1: 0, g2: 1}).astype(int)

    design_df = pd.DataFrame(
        {
            "groups": groups01,
            "age": pd.to_numeric(d_sub["age"], errors="coerce"),
            "SEX": d_sub["SEX"],
            "Field Strength": d_sub["Field Strength"],
        },
        index=d_sub.index,
    )

    if design_df["SEX"].dtype == object:
        design_df["SEX"] = pd.Categorical(design_df["SEX"]).codes

    design_df["SEX"] = pd.to_numeric(design_df["SEX"], errors="coerce")

    # encode field strength as a binary categorical covariate: 1.5T -> 0, 3T -> 1
    design_df["Field Strength"] = pd.to_numeric(design_df["Field Strength"], errors="coerce")
    design_df["Field Strength"] = design_df["Field Strength"].map({1.5: 0, 3.0: 1})

    return design_df

def run_group_comparisons(
        Y_aligned: pd.DataFrame,
        design: pd.DataFrame,
        ref_df: pd.DataFrame,
        contrasts: dict[str, tuple[float, float]],
        labels: dict[float, str],
        n_perm: int = 10000,
) -> tuple[pd.DataFrame, dict]:
    """Run NiSpace group comparisons for all requested contrasts"""
    all_rows = []
    outputs = {}

    for contrast_name, (g1, g2) in contrasts.items():
        mask = design["CONCOHORT"].astype(float).isin([g1, g2])

        y_sub = Y_aligned.loc[mask].copy()
        d_sub = design.loc[mask].copy()

        design_df = build_nispace_design(d_sub, g1, g2)

        keep = ~(
            y_sub.isna().any(axis=1)
            | design_df[["groups", "age", "SEX", "Field Strength"]].isna().any(axis=1)
        )

        y_sub = y_sub.loc[keep]
        design_df = design_df.loc[keep]
        d_sub = d_sub.loc[keep]

        print ("\n--------------------------")
        print(f"Contrast: {contrast_name}")
        print("Counts:", d_sub["CONCOHORT"].astype(float).map(labels).value_counts().to_dict())
        print("Y shape", y_sub.shape)
        print("Design shape:", design_df.shape)
        print("Reference maps:", len(ref_df.index))

        np.random.seed(42)

        colocs, pvals, qvals, nsp = group_comparison(
            y=y_sub,
            x=ref_df,
            parcellation="DesikanKilliany",
            design=design_df,
            comparison_method="hedges(a,b)",
            colocalization_method="spearman",
            n_perm=n_perm,
            n_proc=-1,
            verbose=True,
        )

        outputs[contrast_name] = {
            "colocs": colocs,
            "p": pvals,
            "q": qvals,
            "nsp": nsp,
        }

        rho = np.asarray(colocs).ravel()
        p = np.asarray(pvals).ravel()
        q = np.asarray(qvals).ravel()

        df_out = pd.DataFrame(
            {
                "reference_map": ref_df.index,
                "rho": rho,
                "p": p,
                "q": q,
                "contrast": contrast_name
            }
        )

        all_rows.append(df_out)

    df_all = pd.concat(all_rows, ignore_index=True).sort_values(["contrast", "q", "p"])

    return df_all, outputs

In [ ]:
#4. Load Data
df = pd.read_csv(DATA_PATH, low_memory=False)

df.head(5)

In [ ]:
print(df["Field Strength"].value_counts(dropna=False))

In [ ]:
Y, design = prepare_brain_and_design(df)

print("\nY shape:", Y.shape)
print("Design shape:", design.shape)
print("\nCONCOHORT counts:")
print(design["CONCOHORT"].value_counts(dropna=False))

In [ ]:
#5. Fetch and select reference maps
df_reference_desikan = fetch_reference(
    "pet",
    collection="UniqueTracers",
    parcellation="DesikanKilliany",
    print_references=True,
)

print("\nFull reference shape:", df_reference_desikan.shape)
print("Available reference maps:")
print(df_reference_desikan.index.tolist())


In [ ]:
df_reference_selected = df_reference_desikan[
    df_reference_desikan.index.get_level_values("map").isin(SELECTED_REFERENCE_MAPS)
]

if SELECTED_REFERENCE_CONTAINS:
    mask = df_reference_selected.index.get_level_values("map").str.contains(
        "|".join(SELECTED_REFERENCE_CONTAINS),
        case=False,
        na=False
    )
    df_reference_selected = df_reference_selected[mask]

reference_table = df_reference_selected.index.to_frame(index=False)
reference_table.columns = ["Set", "Map"]
reference_table = reference_table.reset_index(drop=True)
reference_table.index = reference_table.index + 1

print("\nSelected reference shape:", df_reference_selected.shape)
print("Selected reference maps:")
print(reference_table.to_string())

In [ ]:
#6. Align Y to reference parcels
Y_aligned = align_y_to_reference(Y, df_reference_selected)

common_idx = Y_aligned.index.intersection(design.index)
Y_aligned = Y_aligned.loc[common_idx].copy()
design = design.loc[common_idx].copy()

In [ ]:
# Save subject-level aligned DK table for R/ggseg visualization
df_ggseg = (
    design.loc[common_idx, ["CONCOHORT", "age", "SEX", "Field Strength"]]
    .copy()
    .assign(PATNO=common_idx)
)

df_ggseg = pd.concat(
    [
        df_ggseg[["PATNO", "CONCOHORT", "age", "SEX", "Field Strength"]],
        Y_aligned
    ],
    axis=1
)

df_ggseg.to_csv("../../data/df1_id_thick_aligned.csv", index=False)
print("Saved:", "../../data/df1_id_thick_aligned.csv")
print(df_ggseg.shape)
display(df_ggseg.head())

## correct place to save because:
## my R ggseg code atm needs: one row per subject, a concohort column, parcel columns

In [ ]:
#7. (optional) parcelwise t-test example
parcelwise_ttests = {}

for contrast_name, (g1, g2) in CONTRASTS.items():
    df_ttest = run_parcelwise_ttest(Y_aligned, design, g1, g2)
    df_ttest_reset = df_ttest.reset_index().rename(columns={"index": "parcel"})

    outname = f"../../results/{contrast_name.lower()}_parcelwise_ttest.csv"
    df_ttest_reset.to_csv(outname, index=False)

    parcelwise_ttests[contrast_name] = df_ttest_reset

    print(f"Saved: {outname}")
    display(df_ttest_reset.head())

    print(f"\nTop 5 parcels by p-value ({contrast_name}):")
    print(df_ttest.sort_values("pvalue").head(5))

    view_surf(
        df_ttest.loc[df_ttest["hemi"] == "L", "Tvalue"],
        parcellation="DesikanKilliany",
        template="fsaverage",
        hemi="L",
    )

    view_surf(
        df_ttest.loc[df_ttest["hemi"] == "R", "Tvalue"],
        parcellation="DesikanKilliany",
        template="fsaverage",
        hemi="R",
    )

In [ ]:
#8. Running NiSpace group comparisons
df_all, outputs = run_group_comparisons(
    Y_aligned=Y_aligned,
    design=design,
    ref_df=df_reference_selected,
    contrasts=CONTRASTS,
    labels=GROUP_LABELS,
    n_perm=N_PERM
)

In [ ]:
#9. Checking the results

print("\n=============================")
print("Combined results")
print("\n=============================")
display(df_all.head(10))

In [ ]:
df_all.to_csv("../../results/nispace_group_comparison_results_shi.csv", index=False)

In [ ]:
for contrast_name, res in outputs.items():
    print("\n=============================")
    print(f"Contrast: {contrast_name}")
    print("\n=============================")

    colocs = res["colocs"]
    pvals = res["p"]
    qvals = res["q"]

    print("Top 5 lowest q-values")
    print(qvals.T["mean"].sort_values().head(5))

    display("Colocalization:", colocs)
    display("p values:", pvals)
    display("q values:", qvals)

print("\nTotal NaNs in selected reference:", df_reference_selected.isna().sum().sum())
print("NaNs per selected map:")
display(df_reference_selected.isna().sum(axis=1).sort_values(ascending=False))

for contrast_name in df_all["contrast"].unique():
    print(f"\nTop 5 results for {contrast_name}:")
    display(
        df_all[df_all["contrast"] == contrast_name]
        .sort_values("q").head(5)
    )

## Advanced analyses: Group Comparison using single-subject z-scores
- **To interpret it correctly:**
1. **What zscore(a, b) means here**
    - It uses group b as the reference (controls) to compute z-scores
    - with our encoding "groups: 0=g1, 1=g2", that means: a= group 0(g1), b= group 1(g2)
    - so for PD_vs_HC with (g1, g2) = (PD, HC) you get PD z-scored relative to HC - which is the "standard" JuSPace style
2. **Make sure the "control" is always the second group**
    - Your contrasts alredy to that for *_vs_HC (HC is g2)
    - for PD_vs_Prodromal, neither is a true "healthy control"; the z-score will still compute, but interpret it as **PD standardized relative to Prodromal**, not "abnormality vs healthy"

In [ ]:
zscore_outputs = {}

for contrast_name, (gA, gB) in CONTRASTS.items():
    mask = design["CONCOHORT"].astype(float).isin([gA, gB])
    y = Y_aligned.loc[mask].copy()
    d = design.loc[mask].copy()

    keep = ~(y.isna().any(axis=1) | d[["age", "SEX", "Field Strength"]].isna().any(axis=1))
    y = y.loc[keep]
    d = d.loc[keep]

    design_sub = build_nispace_design(d, gA, gB)

    print(f"\n{contrast_name}")
    print("Counts:", d["CONCOHORT"].astype(float).map(GROUP_LABELS).value_counts().to_dict())

    colocs, pvals, qvals, nsp = group_comparison(
        y=y,
        x=df_reference_selected,
        parcellation="DesikanKilliany",
        design=design_sub,
        comparison_method="zscore(a,b)",
        colocalization_method="spearman",
        n_perm=N_PERM,
        n_proc=-1,
        verbose=False,
        plot_design=False,
    )

    zscore_outputs[contrast_name] = {
        "colocs": colocs,
        "p": pvals,
        "q": qvals,
        "nsp": nsp,
    }

- The NiSpace z-score group comparison produces:
    - One colocalization valye per reference map per contrast
    - plus null/permutation significance
    - DOES NOT PRODUCE --> one rho per subject per ma 

"This is the same NiSPace z-score colocalization analysis as the Destriuex example, adapted to the DK atlas and run separately from each parirwise group contrast with covariate adjustment"

In [ ]:
for contrast, res in zscore_outputs.items():
    print(f"\n{contrast}")
    display("Colocalization:", res["colocs"].head(3))
    display("P values:", res["p"].head())
    display("Q values:", res["q"].head())

In [ ]:
import pandas as pd

all_rows = []

for contrast, res in zscore_outputs.items():
    colocs = res["colocs"].copy()

    # make sure subject index has a name
    if colocs.index.name is None:
        colocs.index.name = "PATNO"

    # stack all column levels into rows
    colocs_long = colocs.stack(list(range(colocs.columns.nlevels))).reset_index()

    # rename the stacked value column
    colocs_long = colocs_long.rename(columns={0: "colocalization"})

    # inspect once if needed
    # print(colocs_long.head())
    # print(colocs_long.columns)

    # build a single map label from all non-PATNO, non-colocalization columns
    meta_cols = [c for c in colocs_long.columns if c not in ["PATNO", "colocalization"]]

    colocs_long["map"] = colocs_long[meta_cols].astype(str).agg(" | ".join, axis=1)

    # keep only the relevant columns
    colocs_long = colocs_long[["PATNO", "map", "colocalization"]].copy()
    colocs_long["contrast"] = contrast

    all_rows.append(colocs_long)

nispace_df = pd.concat(all_rows, ignore_index=True)

In [ ]:
print(nispace_df.head())
print(nispace_df.shape)
print(nispace_df["contrast"].unique())

In [ ]:
clinical_df = df[["PATNO", "updrs3_score", "moca", "gds", "SEX", "age"]].copy()

In [ ]:
nispace_df["PATNO"] = nispace_df["PATNO"].astype(str)
clinical_df["PATNO"] = clinical_df["PATNO"].astype(str)

In [ ]:
merged_df = nispace_df.merge(clinical_df, on="PATNO", how="inner")

In [ ]:
print(merged_df.head())
print(merged_df.shape)

In [ ]:
# 10. Save merged dataset
merged_df.to_csv("../../data/merged_df_thickness_cortical.csv", index=False)

print("\nSaved as merged_df_thickness_cortical.csv")

## BREAKING HERE AND CREATING A NEW NOTEBOOK TO CHECK CLINICAL CORRELATION BY Z-SCORE